In [114]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from googletrans import Translator
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt 
import re
import time
import os

chromedriver_path = "/opt/homebrew/bin/chromedriver"
service = Service(chromedriver_path)
options = Options()

options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-extensions")
options.add_argument("--profile-directory=Default")
options.add_argument("--disable-infobars")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")

driver = None  
analyzer = SentimentIntensityAnalyzer()

In [115]:
def get_driver():
    """
    Menginisialisasi driver hanya saat diperlukan.
    """
    global driver
    if not driver:
        driver = webdriver.Chrome(service=service, options=options)
    return driver

def scrape_all_video_links(url):
    """
    Mengambil semua link video dari halaman TikTok pengguna dengan scrolling.
    """
    if not url:
        print("URL tidak valid. Silakan masukkan URL yang benar.")
        return []
    
    driver = get_driver()
    driver.get(url)
    print("Silakan selesaikan CAPTCHA jika muncul.")
    input("Tekan ENTER setelah Anda menyelesaikan CAPTCHA...")
    
    SCROLL_PAUSE_TIME = 3
    last_height = driver.execute_script("return document.body.scrollHeight")
    
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(SCROLL_PAUSE_TIME)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    video_links = [link["href"] for link in soup.find_all("a", href=True) if "/video/" in link["href"]]
    video_links = list(set(video_links))
    print(f"Jumlah video ditemukan: {len(video_links)}")
    
    return video_links

def scrape_tiktok_comments(url):
    """
    Mengambil semua komentar dari satu video TikTok dengan waktu tunggu.
    """
    if not url:
        print("URL tidak valid. Silakan masukkan URL yang benar.")
        return []

    try:
        driver = get_driver()
        driver.get(url)
        time.sleep(5)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        comment_elements = soup.find_all("span", class_="TUXText TUXText--tiktok-sans TUXText--weight-normal")
        comments = [element.get_text() for element in comment_elements]
        print(f"Jumlah komentar ditemukan di {url}: {len(comments)}")
        return comments
    except Exception as e:
        print(f"Terjadi error saat mengambil komentar: {e}")
        input("Tekan ENTER untuk mencoba lagi...")
        return scrape_tiktok_comments(url)
    
def load_video_links_from_csv(csv_file):
    """
    Memuat link video dari file CSV jika ada.
    """
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        return df["VideoLink"].tolist()
    return []

def save_video_links_to_csv(video_links, csv_file):
    """
    Menyimpan link video ke file CSV.
    """
    df = pd.DataFrame({"VideoLink": video_links})
    df.to_csv(csv_file, index=False)
    print(f"Link video disimpan ke '{csv_file}'.")

def load_processed_comments(csv_file):
    """
    Memuat komentar yang sudah diproses dari file CSV jika ada.
    """
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        return df
    return pd.DataFrame(columns=["VideoLink", "Comment"])

def save_comments_to_csv(df, csv_file):
    """
    Menyimpan komentar yang sudah diproses ke file CSV.
    """
    df.to_csv(csv_file, index=False)
    print(f"Komentar disimpan ke '{csv_file}'.")

def translate(comment):
    """
    Menerjemahkan komentar dari bahasa Bali ke bahasa Indonesia menggunakan Google Translate.
    """
    # translator = Translator()
    # comment = translator.translate(comment, src='id', dest='en').text
    # print(comment)
    
    driver = get_driver()
    driver.get("https://translate.google.com")

    source_dropdown = driver.find_element("xpath", '//button[@aria-label="More source languages"]')
    source_dropdown.click()
    time.sleep(1)  # Tunggu dropdown muncul
    
    # Pilih Bahasa Bali dari daftar
    bali_option = driver.find_element("xpath", '//div[text()="Balinese"]')
    bali_option.click()
    time.sleep(1)
    
    target_dropdown = driver.find_element("xpath", '//button[@aria-label="More target languages"]')
    target_dropdown.click()
    time.sleep(2)  # Tunggu dropdown muncul
    
    # Ketik Bahasa indonesia
    indonesia_search = driver.find_element("xpath", '//textarea[@aria-label="Search languages"]')
    indonesia_search.send_keys("Indonesian")
    time.sleep(1)
    
    # Pilih Bahasa indonesia dari daftar
    indonesia_option = driver.find_element("xpath", '//div[@data-language-code="id"]')
    indonesia_option.click()
    time.sleep(1)
    
    input_box = driver.find_element("xpath", '//textarea[@aria-label="Source text"]')
    input_box.send_keys(comment)
    
    # Ambil hasil terjemahan
    time.sleep(2)
    output_box = driver.find_element("xpath", '//div[@class="lRu31"]')
    translated_text = output_box.text
    
    return translated_text

def preprocessing(comments):
    """
    Membersihkan komentar, termasuk menghapus URL, angka, emoji, karakter khusus,
    spasi ganda, dan huruf berulang.
    """
    processed_comments = [] 
    translate_result = [] 
    
    for comment in comments:
        # Hapus URL
        comment = re.sub(r"http\S+|www\S+|https\S+", "", comment, flags=re.MULTILINE)
        # Hapus angka
        comment = re.sub(r"\d+", "", comment)
        # Hapus emoji dan karakter khusus
        comment = re.sub(r"[^\w\s,]", " ", comment)
        # Hapus spasi ganda
        comment = re.sub(r"\s+", " ", comment).strip()
        # Hapus huruf yang berulang
        # comment = re.sub(r"(.)\1+", r"\1", comment)
        
        if comment.strip():  # Pastikan komentar tidak kosong setelah diproses
            processed_comments.append(comment)
            translate_word = translate(comment)
            translate_result.append(translate_word)
    
    df = pd.DataFrame({
        "Comment" : processed_comments,
        "Translate" : translate_result
    })
    
    return df

def vader(comments):
    sentiments = [] 
    
    for comment in comments:
        scores = analyzer.polarity_scores(comment)
        sentiment = "Netral"
    
        if scores['compound'] > 0.05:
            sentiment = "Positif"
        elif scores['compound'] < -0.05:
            sentiment = "Negatif"
        
        sentiments.append(sentiment)
    
    return sentiments
    
def start(url, name_user):
    print("Memeriksa file CSV untuk link video...")
    video_links = load_video_links_from_csv(f"{name_user}_video.csv")
    
    if not video_links:
        print("Tidak ada link video di file CSV. Mengambil link dari halaman pengguna TikTok...")
        video_links = scrape_all_video_links(url)
        save_video_links_to_csv(video_links, f"{name_user}_video.csv")
    else:
        print(f"Memuat {len(video_links)} link video dari file CSV.")
    
    processed_df = load_processed_comments(f"{name_user}_comment.csv")
    processed_video_links = processed_df["VideoLink"].unique()
    
    for video_link in video_links:
        if video_link in processed_video_links:
            print(f"Komentar dari video {video_link} sudah diambil sebelumnya, melewati...")
            continue
        
        print(f"Mengambil komentar dari video: {video_link}")
        comments = scrape_tiktok_comments(video_link)

        if comments:
            temp_df = pd.DataFrame({
                "VideoLink": [video_link] * len(comments),
                "Comment": comments
            })
            processed_df = pd.concat([processed_df, temp_df], ignore_index=True)
            save_comments_to_csv(processed_df, f"{name_user}_comment.csv")
    
    print("Proses selesai. Komentar disimpan.")

def sentiment(file_comment):
    df = load_processed_comments(file_comment)
    df = df.dropna(subset=["Comment"]).reset_index(drop=True)
    df["Comment"] = df["Comment"].astype(str)
    fix_comment = preprocessing(df["Comment"].tolist())
    fix_comment

sentiment("gadjah_comment.csv")

NoSuchElementException: Message: no such element: Unable to locate element: {"method":"xpath","selector":"//textarea[@aria-label="Search languages"]"}
  (Session info: chrome=131.0.6778.86); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
0   chromedriver                        0x0000000104c6bac4 cxxbridge1$str$ptr + 3651580
1   chromedriver                        0x0000000104c64314 cxxbridge1$str$ptr + 3620940
2   chromedriver                        0x00000001046cc4b4 cxxbridge1$string$len + 89224
3   chromedriver                        0x0000000104710898 cxxbridge1$string$len + 368748
4   chromedriver                        0x000000010474a0fc cxxbridge1$string$len + 604368
5   chromedriver                        0x00000001047050b0 cxxbridge1$string$len + 321668
6   chromedriver                        0x0000000104705d00 cxxbridge1$string$len + 324820
7   chromedriver           